In [98]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split
from IPython.display import display
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [6]:
!curl -L -o data-homework-3.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0   924k      0 --:--:-- --:--:-- --:--:--  922k


In [7]:
df = pd.read_csv('data-homework-3.csv')

In [8]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [12]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [58]:
numerical = df.dtypes.index[(df.dtypes == 'float64') | (df.dtypes == 'int64')]

In [15]:
categorical = df.dtypes.index[df.dtypes == 'object']

In [44]:
df.industry = df.industry.fillna('NA')

In [45]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [46]:
df[categorical] = df[categorical].fillna('NA')

In [51]:
for col in categorical:
    print(df[col].value_counts(dropna=False).to_string(), "\n")

lead_source
organic_search    1251
paid_ads          1140
social_media      1055
referral           790
events             616
NA                 148 

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240 

employment_status
employed         2633
student           760
self_employed     715
unemployed        698
NA                194 

location
north_america    1371
europe           1149
asia             1052
south_america     749
africa            471
NA                208 



In [59]:
numerical

Index(['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [60]:
df[numerical] = df[numerical].fillna(0.0)

In [68]:
for i in numerical:
    print(i)
    print(df[numerical].corrwith(df[i]).drop(i).round(3), '\n')

annual_income
number_of_courses_viewed    0.161
interaction_count           0.123
lead_score                  0.229
converted                   0.190
dtype: float64 

number_of_courses_viewed
annual_income        0.161
interaction_count    0.722
lead_score           0.757
converted            0.395
dtype: float64 

interaction_count
annual_income               0.123
number_of_courses_viewed    0.722
lead_score                  0.916
converted                   0.449
dtype: float64 

lead_score
annual_income               0.229
number_of_courses_viewed    0.757
interaction_count           0.916
converted                   0.483
dtype: float64 

converted
annual_income               0.190
number_of_courses_viewed    0.395
interaction_count           0.449
lead_score                  0.483
dtype: float64 



In [69]:
cor = df[numerical].corr()
cor

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [240]:
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [241]:
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42)

In [242]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [243]:
mi = df_train[categorical].apply(lambda x: mutual_info_score(x, df_train.converted)).round(2)

In [244]:
mi = mi.sort_values(ascending=False)
mi

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [245]:
y_train = df_train.converted
y_val = df_val.converted
y_test = df_test.converted

In [246]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [247]:
dv = DictVectorizer(sparse=False)

In [248]:
dicts_train = df_train.to_dict(orient='records')
dicts_val = df_val.to_dict(orient='records')
dicts_test = df_test.to_dict(orient='records')

In [249]:
dv.fit(dicts_train)

,dtype,<class 'numpy.float64'>
,separator,'='
,sparse,False
,sort,True


In [250]:
X_train = dv.transform(dicts_train)

In [251]:
X_val = dv.transform(dicts_val)
X_test = dv.transform(dicts_test)

In [252]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [253]:
y_pred = model.predict_proba(X_train)[:,1]

In [254]:
decision = (y_pred >=0.5)
(decision == y_train).mean()

np.float64(0.625)

In [255]:
y_pred_val = model.predict_proba(X_val)[:,1]
decision_val = (y_pred_val >=0.5)
(decision_val == y_val).mean()

np.float64(0.645)

In [256]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [258]:
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train,y_train)
    y_pred = model.predict_proba(X_train)[:,1]
    y_pred_val = model.predict_proba(X_val)[:,1]
    decision_val = (y_pred_val >=0.5)
    print((decision_val == y_val).mean())

0.598
0.598
0.613
0.645


In [236]:
X_train_lead_source = dv.fit_transform(df_train.drop(columns=['lead_source']).to_dict(orient='records'))
X_val_lead_source = dv.transform(df_val.drop(columns=['lead_source']).to_dict(orient='records'))
model.fit(X_train_lead_source, y_train)
y_pred_lead_source = model.predict_proba(X_val_lead_source)[:,1]
decision_lead_source = (y_pred_lead_source >=0.5)
(decision_lead_source == y_val).mean()

np.float64(0.642)

In [237]:
X_train_number_of_courses_viewed = dv.fit_transform(df_train.drop(columns=['number_of_courses_viewed']).to_dict(orient='records'))
X_val_number_of_courses_viewed = dv.transform(df_val.drop(columns=['number_of_courses_viewed']).to_dict(orient='records'))
model.fit(X_train_number_of_courses_viewed, y_train)
y_pred_number_of_courses_viewed = model.predict_proba(X_val_number_of_courses_viewed)[:,1]
decision_number_of_courses_viewed = (y_pred_number_of_courses_viewed >=0.5)
(decision_number_of_courses_viewed == y_val).mean()

np.float64(0.643)

In [238]:
X_train_interaction_count = dv.fit_transform(df_train.drop(columns=['interaction_count']).to_dict(orient='records'))
X_val_interaction_count = dv.transform(df_val.drop(columns=['interaction_count']).to_dict(orient='records'))
model.fit(X_train_interaction_count, y_train)
y_pred_interaction_count = model.predict_proba(X_val_interaction_count)[:,1]
decision_interaction_count = (y_pred_interaction_count >=0.5)
(decision_interaction_count == y_val).mean()

np.float64(0.601)